In [1]:
import os
import sys

import torch

_nb = os.getcwd()
if os.path.basename(_nb) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_nb, ".."))
else:
    PROJECT_ROOT = os.environ.get(
        "AUDIO_STREAM_ADAPTER_ROOT",
        os.path.abspath(os.path.join(_nb, "..")),
    )
SRC_ROOT = os.path.join(PROJECT_ROOT, "src")
for _p in (SRC_ROOT, PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from transformers import AutoModelForCausalLM, AutoTokenizer

# Streaming Audio Encoder

In [2]:
# src/utils/qwen_model_loader.py

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

if device == "cuda":
    torch.cuda.init()

print(f"Using device: {device}")

MODEL_ID = "Qwen/Qwen3-8B"

# Load tokenizer and model
qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

_load_kw = dict(torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)
if device == "cuda":
    _load_kw["device_map"] = "auto"

qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_load_kw)
if device == "cuda" and "device_map" not in _load_kw:
    qwen_model = qwen_model.to(device)

qwen_model.eval()

print("Qwen3-8B model ready!")


Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Qwen3-8B model ready!


In [3]:
# src/utils/whisper_model_loader.py

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import pipeline

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")

MODEL_ID = "openai/whisper-small"

# Load processor and model
whisper_processor = WhisperProcessor.from_pretrained(MODEL_ID)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
whisper_model.to(device)

# Build inference pipeline
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

print(type(whisper_model))  # should output: <class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>
print("Whisper small model ready!")

Using device: cuda


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


<class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>
Whisper small model ready!


## Transcription Summarization

In [4]:
print(os.getcwd())

/home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter/notebooks


In [5]:
# !ls -l ../datasets/librispeech_data/LibriSpeech/train-clean-100

In [6]:
import glob

dataset_root = "../datasets/librispeech_data/LibriSpeech/train-clean-100"
flac_files = sorted(glob.glob(f"{dataset_root}/**/*.flac", recursive=True))
print(f"Found {len(flac_files)} files\n")

Found 28539 files



### Transcribe (transcribe.py)

In [7]:
# transcribe_wav and transcribe_dataset_stream
import librosa
from IPython.display import Audio, display

# import pipe from src/utils/whisper_model_loader.py
# from src.utils.whisper_model_loader import pipe

SAMPLES_RATE = 16000

def transcribe_wav(file_path:str) -> str:
    waveform, sample_rate = librosa.load(file_path, sr=None)
    if len(waveform.shape) > 1:
        waveform = librosa.to_mono(waveform)
    if sample_rate != SAMPLES_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLES_RATE)
    result = whisper_pipe(waveform)
    return result["text"], waveform, sample_rate

def transcribe_dataset_stream(dataset_root: str, num_samples: int = 5):
    """
    transcribes files one at a time and yields each result immediately as {"file": path, "transcription": text}.

    start processing results as soon as they arrive, without waiting for all files to finish.
    """
    flac_files = sorted(glob.glob(f"{dataset_root}/**/*.flac", recursive=True))
    print(f"Found {len(flac_files)} files\n")
    count = 0

    for path in flac_files:
        if count < num_samples:
            transcription, waveform, original_sample_rate = transcribe_wav(path)
            # print(f"{path} → {transcription}\n")
            count = count + 1
            yield {"file": path, "transcription": transcription, "waveform": waveform}

In [8]:
def transcribe_dataset(dataset_root: str) -> list[dict]:
    """
    Eagerly transcribes all files and returns a list of
    {"file": path, "transcription": text} dicts.

    Use transcribe_dataset_stream() instead if you want to process
    results incrementally as each file is transcribed.
    """
    return list[dict[str, str]](transcribe_dataset_stream(dataset_root))

# results = transcribe_dataset(dataset_root)
# results[0], display(Audio(results[0]["waveform"], rate=SAMPLES_RATE))

### Qwen Summarization (qwen_summarize.py)

In [9]:
# helper

# Generation settings for Qwen
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.7
DO_SAMPLE = True
# NOTE: With `device_map="auto"` the model can be sharded across CPU/GPU.
# The safe approach is to put inputs on the model's embedding device.

def _qwen_input_device(model) -> torch.device:
    emb = getattr(model, "get_input_embeddings", lambda: None)()
    if emb is not None and getattr(emb, "weight", None) is not None:
        return emb.weight.device
    return next(model.parameters()).device


def qwen_summarize(
    text: str,
    context: str = "audio transcription chunk",
    *,
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
    do_sample: bool = False,
    use_cache: bool = False,
) -> str:
    prompt = (
        f"You are a helpful assistant. Below is a segment of {context}. "
        f"Please provide a concise and accurate summary of the content.\n\n"
        f"Transcription:\n{text}\n\n"
        f"Summary:"
    )

    if hasattr(tokenizer, "apply_chat_template"):
        formatted = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        enc = tokenizer(formatted, return_tensors="pt")
    else:
        enc = tokenizer(prompt, return_tensors="pt")

    enc = enc.to(_qwen_input_device(model))

    pad_token_id = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )

    # Best-effort: free unused cached blocks before each generate.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        with torch.inference_mode():
            output_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=do_sample,
                use_cache=use_cache,
                pad_token_id=pad_token_id,
            )
    except RuntimeError as e:
        # If VRAM is tight, a smaller output often succeeds.
        msg = str(e)
        if "CUBLAS_STATUS_ALLOC_FAILED" in msg or "out of memory" in msg:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            with torch.inference_mode():
                output_ids = model.generate(
                    **enc,
                    max_new_tokens=min(64, max_new_tokens),
                    temperature=0.0,
                    do_sample=False,
                    use_cache=False,
                    pad_token_id=pad_token_id,
                )
        else:
            raise

    generated = output_ids[0][enc["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# for entry in transcribe_dataset_stream(dataset_root, num_samples=2):
#     print(entry["transcription"])
#     print("-"*100)
#     print(qwen_summarize(entry["transcription"]))
#     print("\n\n")
#     break

## Projector

In [10]:
import torch
import torch.nn as nn

class WhisperToQwenProjector(nn.Module):
    """
    Simple linear projection network for Whisper-to-LLM dimension mapping.

    This projector performs frame-by-frame projection, preserving the full
    temporal sequence length. Unlike the StreamingAdapter, it does not provide
    any temporal compression, making it useful as a baseline or when full
    temporal resolution is required.

    Architecture:
        Linear(d_in, 2048) → ReLU activation → Linear(2048, d_out)

    Args:
        in_dim: Input dimension from Whisper encoder (default: 768 for whisper-small)
        out_dim: Output dimension matching LLM embedding (default: 4096 for Qwen3-8B)

    Example:
        >>> projector = WhisperToQwenProjector(in_dim=768, out_dim=4096)
        >>> encoder_output = torch.randn(1, 1500, 768)  # Whisper output
        >>> projected = projector(encoder_output)
        >>> print(projected.shape)  # torch.Size([1, 1500, 4096])
    """

    def __init__(self, in_dim: int = 768, out_dim: int = 4096):
        """
        Initialize the Whisper-to-Qwen projector.

        Args:
            in_dim: Dimension of Whisper encoder output (e.g., 768 for whisper-small)
            out_dim: Dimension of LLM embedding space (e.g., 4096 for Qwen3-8B)
        """
        super().__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim

        # Two-layer projection with ReLU activation
        self.proj = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048, out_dim),
        )

        # Initialize weights for better training stability
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize layer weights using Xavier uniform initialization."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Project Whisper encoder embeddings to LLM embedding space.

        Args:
            x: Input tensor of shape (batch_size, sequence_length, in_dim)
               where sequence_length is typically 1500 frames for 30s of audio

        Returns:
            Projected tensor of shape (batch_size, sequence_length, out_dim)
            The sequence length is preserved through frame-by-frame projection

        Example:
            >>> projector = WhisperToQwenProjector()
            >>> x = torch.randn(2, 1500, 768)  # batch=2, frames=1500
            >>> y = projector(x)
            >>> print(y.shape)  # torch.Size([2, 1500, 4096])
        """
        return self.proj(x)

    def get_num_parameters(self) -> int:
        """
        Get the total number of trainable parameters.

        Returns:
            Total number of parameters
        """
        return sum(p.numel() for p in self.parameters())

In [11]:
whisper_to_qwen_projector = WhisperToQwenProjector()

# Test the projector
test_input = torch.randn(1, 1500, 768)  # Example input shape
projected = whisper_to_qwen_projector(test_input)
print(projected.shape)  # Should print: torch.Size([1, 1500, 4096])

torch.Size([1, 1500, 4096])


## Audio Encoder

In [13]:
# (A) Reference: Whisper's intended decoding path (generate -> decode)

def whisper_transcribe_generate(waveform: torch.Tensor):
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)

    with torch.inference_mode():
        generated_ids = whisper_model.generate(input_features)  # (1, T)

    text = whisper_processor.tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]

    return input_features, generated_ids, text


# for entry in transcribe_dataset_stream(dataset_root, num_samples=1):
#     print(entry["transcription"])
#     print("-" * 100)
#     input_features, generated_ids, text = whisper_transcribe_generate(entry["waveform"])
#     print(generated_ids.shape)
#     print(text)
#     break


In [ ]:
# This reproduces "decoding" using your own loop so you can inspect tokens.

def whisper_manual_greedy_decode(
    waveform: torch.Tensor,
    *,
    language_token: str = "<|en|>",
    task_token: str = "<|transcribe|>",
    max_steps: int = 4096,
):
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
    input_features = inputs.input_features.to(device, dtype=torch_dtype)
    attention_mask = inputs.attention_mask.to(device, dtype=torch_dtype)

    print(input_features.shape, attention_mask.shape)

    print("ATTENTION MASK:")
    print(attention_mask)
    print(attention_mask.sum(dim=1))

    tok = whisper_processor.tokenizer

    start_id = whisper_model.config.decoder_start_token_id
    lang_id = tok.convert_tokens_to_ids(language_token)
    task_id = tok.convert_tokens_to_ids(task_token)

    # Prompt tokens: start-of-transcript + language + task
    prompt_ids = [start_id]
    if lang_id is not None and lang_id != tok.unk_token_id:
        prompt_ids.append(int(lang_id))
    if task_id is not None and task_id != tok.unk_token_id:
        prompt_ids.append(int(task_id))

    decoder_input_ids = torch.tensor([prompt_ids], device=device, dtype=torch.long)

    with torch.inference_mode():
        encoder_outputs = whisper_model.model.encoder(input_features, attention_mask=attention_mask)
        print("encoder_last_hidden_state:", encoder_outputs.last_hidden_state.shape)
        print("Encoder outputs keys:", encoder_outputs.keys())

        print("Encoder output: ", encoder_outputs)

        for step in range(max_steps):
            print(f"\nstep={step}")
            print("decoder_input_ids:", decoder_input_ids.shape)

            decoder_outputs = whisper_model.model.decoder(
                input_ids=decoder_input_ids,
                encoder_hidden_states=encoder_outputs.last_hidden_state,
                use_cache=False,
            )
            print("decoder_last_hidden_state:", decoder_outputs.last_hidden_state.shape)

            logits = whisper_model.proj_out(decoder_outputs.last_hidden_state)  # (1, T, V)
            print("logits:", logits.shape)

            next_token = logits[:, -1, :].argmax(dim=-1)  # (1,)
            print("next_token:", next_token.shape, int(next_token.item()))
            next_id = int(next_token.item())

            decoder_input_ids = torch.cat([decoder_input_ids, next_token[:, None]], dim=1)

            # Stop on EOS from model config (don’t use tokenizer bos/eos here; Whisper tokenizer reuses <|endoftext|>)
            if whisper_model.config.eos_token_id is not None and next_id == whisper_model.config.eos_token_id:
                print("stopping: hit eos_token_id", next_id)
                break

    token_ids = decoder_input_ids[0].tolist()
    text = tok.decode(token_ids, skip_special_tokens=True)
    return token_ids, text


for entry in transcribe_dataset_stream(dataset_root, num_samples=1):
    print(entry["transcription"])
    print("-" * 100)
    token_ids, text = whisper_manual_greedy_decode(entry["waveform"], max_steps=64)
    print("num_tokens:", len(token_ids))
    print(token_ids[:20], "...")
    print(text)
    break


Found 28539 files

 CHAPTER I Mrs. Rachel Lind is surprised. Mrs. Rachel Lind lived just where the Avonlea main road dipped down into a little hollow, fringed with alders and ladies' eardrops and traversed by a brook
----------------------------------------------------------------------------------------------------
torch.Size([1, 80, 3000]) torch.Size([1, 3000])
ATTENTION MASK:
tensor([[1., 1., 1.,  ..., 0., 0., 0.]], device='cuda:0', dtype=torch.float16)
tensor([1409.], device='cuda:0', dtype=torch.float16)
encoder_last_hidden_state: torch.Size([1, 1500, 768])
Encoder outputs keys: odict_keys(['last_hidden_state'])
Encoder output:  BaseModelOutput(last_hidden_state=tensor([[[-0.1115, -2.9258,  0.9189,  ...,  0.2939,  0.4287, -0.8394],
         [ 1.2539, -1.4238,  1.2168,  ...,  0.4954,  0.1655, -0.8120],
         [ 1.3867, -0.9399,  1.9941,  ..., -0.3076,  0.1415, -1.2354],
         ...,
         [ 0.3047, -0.0294,  0.0717,  ..., -0.5581,  0.0394, -0.6016],
         [-0.4207, -0.4617

: 

## Encoder Time Analysis

In [ ]:
import time
import math
import matplotlib.pyplot as plt


def whisper_encode(waveform: torch.Tensor):
    start_time = time.time()
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)

    with torch.inference_mode():
        encoder_outputs = whisper_model.model.encoder(input_features)

    elapsed_s = time.time() - start_time
    enc = encoder_outputs.last_hidden_state  # (1, 1500, 768)

    # One simple scalar summary of the encoder output (to make it plottable)
    enc_rms = float(enc.float().pow(2).mean().sqrt().item())

    return enc, elapsed_s, enc_rms


lengths = []
encode_times_s = []
encoder_rms = []

# for entry in transcribe_dataset_stream(dataset_root, num_samples=100):
#     text = entry["transcription"]
#     L = len(text)
#     enc, t_s, rms = whisper_encode(entry["waveform"])

#     lengths.append(L)
#     encode_times_s.append(t_s)
#     encoder_rms.append(rms)

# print("collected:", len(lengths), "samples")

# fig, ax1 = plt.subplots(figsize=(7, 4))

# ax1.scatter(lengths, encode_times_s, alpha=0.8)
# ax1.set_xlabel("transcription length (chars)")
# ax1.set_ylabel("encoder time (seconds)")
# ax1.grid(True, alpha=0.25)

# ax2 = ax1.twinx()
# ax2.plot(lengths, encoder_rms, "o", alpha=0.5, color="tab:orange")
# ax2.set_ylabel("encoder output RMS")

# plt.title("Whisper encoder: length vs time / RMS")
# plt.tight_layout()
# plt.show()

## Q-former Adapter

In [ ]:
#  src/adapter/cross_attention.py

class QFormerLayer(nn.Module):
    """
    Single Q-Former layer following BLIP-2 architecture.

    Sub-layer 1 — Self-Attention:
        Queries attend to each other. This is what makes Q-Former different
        from plain cross-attention. Without it, each query works independently
        and they may extract redundant information. With self-attention,
        Query 0 can see what Query 1 is capturing and focus elsewhere.

    Sub-layer 2 — Cross-Attention:
        Queries attend to encoder frames (Whisper output). This is where
        the actual information extraction happens.

    Sub-layer 3 — Feed-Forward Network:
        Standard transformer FFN for per-token nonlinear transformation.

    Args:
        d_model: Dimension of queries and output.
        num_heads: Number of attention heads (shared across self and cross attn).
        d_ffn: Hidden dimension of the feed-forward network.
        dropout: Dropout rate.
    """

    def __init__(
        self,
        d_model: int = 1024,
        num_heads: int = 4,
        d_ffn: int = 2048,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.scale = math.sqrt(self.d_head)

        # ---- Sub-layer 1: Self-Attention (queries ↔ queries) ----
        self.self_attn_q = nn.Linear(d_model, d_model)
        self.self_attn_k = nn.Linear(d_model, d_model)
        self.self_attn_v = nn.Linear(d_model, d_model)
        self.self_attn_o = nn.Linear(d_model, d_model)
        self.norm_self = nn.LayerNorm(d_model)
        self.self_attn_dropout = nn.Dropout(dropout)

        # ---- Sub-layer 2: Cross-Attention (queries → encoder frames) ----
        self.cross_attn_q = nn.Linear(d_model, d_model)
        self.cross_attn_k = nn.Linear(d_model, d_model)
        self.cross_attn_v = nn.Linear(d_model, d_model)
        self.cross_attn_o = nn.Linear(d_model, d_model)
        self.norm_cross_q = nn.LayerNorm(d_model)
        self.norm_cross_kv = nn.LayerNorm(d_model)
        self.cross_attn_dropout = nn.Dropout(dropout)

        # ---- Sub-layer 3: Feed-Forward Network ----
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
            nn.Dropout(dropout),
        )
        self.norm_ffn = nn.LayerNorm(d_model)

    def _multihead_attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        dropout: nn.Dropout,
    ) -> torch.Tensor:
        """
        Shared multi-head attention logic for both self and cross attention.

        Args:
            q: (batch, seq_q, d_model) — already projected queries
            k: (batch, seq_k, d_model) — already projected keys
            v: (batch, seq_k, d_model) — already projected values
            dropout: dropout module for attention weights

        Returns:
            (batch, seq_q, d_model)
        """
        batch_size, seq_q, _ = q.shape
        seq_k = k.shape[1]

        # Reshape to multi-head: (batch, seq, d_model) → (batch, heads, seq, d_head)
        q = q.view(batch_size, seq_q, self.num_heads, self.d_head).transpose(1, 2)
        k = k.view(batch_size, seq_k, self.num_heads, self.d_head).transpose(1, 2)
        v = v.view(batch_size, seq_k, self.num_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = dropout(attn_weights)

        # Weighted sum and merge heads
        output = torch.matmul(attn_weights, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_q, self.d_model)

        return output

    def forward(
        self,
        queries: torch.Tensor,
        encoder_features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            queries: (batch, m, d_model) — learnable query vectors
            encoder_features: (batch, T, d_model) — Whisper encoder output

        Returns:
            (batch, m, d_model) — updated query representations
        """
        # ---- Sub-layer 1: Self-Attention (queries attend to each other) ----
        q_norm = self.norm_self(queries)
        sa_out = self._multihead_attention(
            q=self.self_attn_q(q_norm),
            k=self.self_attn_k(q_norm),
            v=self.self_attn_v(q_norm),
            dropout=self.self_attn_dropout,
        )
        sa_out = self.self_attn_o(sa_out)
        queries = queries + sa_out  # residual

        # ---- Sub-layer 2: Cross-Attention (queries attend to encoder) ----
        q_norm = self.norm_cross_q(queries)
        kv_norm = self.norm_cross_kv(encoder_features)
        ca_out = self._multihead_attention(
            q=self.cross_attn_q(q_norm),
            k=self.cross_attn_k(kv_norm),
            v=self.cross_attn_v(kv_norm),
            dropout=self.cross_attn_dropout,
        )
        ca_out = self.cross_attn_o(ca_out)
        queries = queries + ca_out  # residual

        # ---- Sub-layer 3: FFN ----
        queries = queries + self.ffn(self.norm_ffn(queries))  # residual

        return queries

In [ ]:
class QFormerAdapter(nn.Module):
    """
    Q-Former Adapter for Whisper model.

    This adapter uses a Q-Former architecture with cross-attention to adapt
    the Whisper model for audio-to-text tasks.
    
    Args:
        d_encoder: Dimension of Whisper encoder output
        num_queries: Number of learnable query vectors
        num_layers: Number of Q-Former layers
        num_heads: Number of attention heads
        d_ffn: Hidden dimension of the feed-forward network
        dropout: Dropout rate   
    """

    def __init__(
        self,
        d_encoder: int = 1024,
        num_queries: int = 4,
        num_heads: int = 4,
        d_ffn: int = 2048,
        dropout: float = 0.1,
        num_layers: int = 2,
    ):
        super().__init__()

        # Learnable query vectors Q ∈ R^{m × D_q}
        self.queries = nn.Parameter(torch.randn(1, num_queries, d_encoder) * 0.02)

        # Q-Former layers
        self.qformer_layers = nn.ModuleList([
            QFormerLayer(d_encoder, num_heads, d_ffn, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, encoder_features: torch.Tensor) -> torch. Tensor:
        """
        Args:
            encoder_features: (batch, T, d_model) — Whisper encoder output

        Returns:
            (batch, m, d_model) — updated query representations
        """

        batch_size = encoder_features.shape[0]

        # Expand learnable queries to batch size
        queries = self.queries.expand(batch_size, -1, -1) # (batch, m, d_model)
        
        for layer in self.qformer_layers:
            queries = layer(queries, encoder_features)

        return queries

## Using Adapter

In [ ]:
q_former_adapter = QFormerAdapter(
    d_encoder=768,
    num_queries=1490, # for 1500 frames
    num_layers=8,
    num_heads=12,
    d_ffn=2048,
    dropout=0.1,
).to(device=device, dtype=torch_dtype)

# Load adapter weights from training checkpoint
ckpt_path = "checkpoints/head_12_layers_8.pt"  # saved by adapter_training.ipynb
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt.get("model_state_dict", ckpt)
missing, unexpected = q_former_adapter.load_state_dict(state_dict, strict=False)
print(f"Loaded checkpoint: {ckpt_path}")
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")

q_former_adapter.eval()

def whisper_manual_greedy_decode(
    waveform: torch.Tensor,
    *,
    language_token: str = "<|en|>",
    task_token: str = "<|transcribe|>",
    max_steps: int = 4096,
):
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)

    tok = whisper_processor.tokenizer

    start_id = whisper_model.config.decoder_start_token_id
    lang_id = tok.convert_tokens_to_ids(language_token)
    task_id = tok.convert_tokens_to_ids(task_token)

    # Prompt tokens: start-of-transcript + language + task
    prompt_ids = [start_id]
    if lang_id is not None and lang_id != tok.unk_token_id:
        prompt_ids.append(int(lang_id))
    if task_id is not None and task_id != tok.unk_token_id:
        prompt_ids.append(int(task_id))

    decoder_input_ids = torch.tensor([prompt_ids], device=device, dtype=torch.long)

    with torch.inference_mode():
        encoder_outputs = whisper_model.model.encoder(input_features)

        print("Before compression: encoder_last_hidden_state:", encoder_outputs.last_hidden_state.shape)
        encoder_outputs = q_former_adapter(encoder_outputs.last_hidden_state)

        print("After compression: encoder_last_hidden_state:", encoder_outputs.shape)

        for step in range(max_steps):
            print(f"\nstep={step}")
            print("decoder_input_ids:", decoder_input_ids.shape)

            decoder_outputs = whisper_model.model.decoder(
                input_ids=decoder_input_ids,
                encoder_hidden_states=encoder_outputs,
                use_cache=False,
            )
            print("decoder_last_hidden_state:", decoder_outputs.last_hidden_state.shape)

            logits = whisper_model.proj_out(decoder_outputs.last_hidden_state)  # (1, T, V)
            print("logits:", logits.shape)

            next_token = logits[:, -1, :].argmax(dim=-1)  # (1,)
            print("next_token:", next_token.shape, int(next_token.item()))
            next_id = int(next_token.item())

            decoder_input_ids = torch.cat([decoder_input_ids, next_token[:, None]], dim=1)

            # Stop on EOS from model config (don’t use tokenizer bos/eos here; Whisper tokenizer reuses <|endoftext|>)
            if whisper_model.config.eos_token_id is not None and next_id == whisper_model.config.eos_token_id:
                print("stopping: hit eos_token_id", next_id)
                break

    token_ids = decoder_input_ids[0].tolist()
    text = tok.decode(token_ids, skip_special_tokens=True)
    return token_ids, text


for entry in transcribe_dataset_stream(dataset_root, num_samples=1):
    print(entry["transcription"])
    print("-" * 100)
    token_ids, text = whisper_manual_greedy_decode(entry["waveform"], max_steps=64)
    print("num_tokens:", len(token_ids))
    print(token_ids[:20], "...")
    print(text)
    break